In [2]:
!pip install scikeras

In [4]:
!pip install "scikit-learn<1.6" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 11.8 MB/s eta 0:00:00
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 3.6.0
    Uninstalling threadpoolctl-3.6.0:
      Successfully uninstalled threadpoolctl-3.6.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: joblib
    Found existing installation: joblib 1.5.3
    Uninstalling joblib-1.5.3:
      Successfully uninstalled joblib-1.5.3
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfu

In [6]:
"""
CS3807 - Deep Learning Laboratory
Experiment 2: Implementation of a Multi-Layer Perceptron (MLP)
for Multi-Class Image Classification on Fashion-MNIST

Covers:
  Task 1 - Dataset Exploration
  Task 2 - Data Preprocessing
  Task 3 - Model Construction
  Task 4 - Model Training
  Task 5 - Model Evaluation
  Section 7 - Hyperparameter Optimization (manual Randomized Search
              with K-Fold cross-validation, implemented directly in
              Keras -- no SciKeras dependency, to avoid the SciKeras /
              scikit-learn "__sklearn_tags__" version-compatibility bug)
  Section 8 - All 9 mandatory plots (saved as PNG files for the report)
"""

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import KFold, ParameterSampler

# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Directory to save all plots for the LaTeX report
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# =======================================================================
# TASK 1: DATASET EXPLORATION
# =======================================================================
print("=" * 70)
print("TASK 1: Dataset Exploration")
print("=" * 70)

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras.datasets.fashion_mnist.load_data()

print(f"Training images shape : {x_train_raw.shape}")
print(f"Training labels shape : {y_train_raw.shape}")
print(f"Testing images shape  : {x_test_raw.shape}")
print(f"Testing labels shape  : {y_test_raw.shape}")
print(f"Number of classes     : {len(np.unique(y_train_raw))}")

# --- Plot 1: Sample Images ---
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_raw[i], cmap="gray")
    ax.set_title(CLASS_NAMES[y_train_raw[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("Sample Images from Fashion-MNIST")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/01_sample_images.png", dpi=200)
plt.close()

# --- Plot 2: Class Distribution ---
unique, counts = np.unique(y_train_raw, return_counts=True)
plt.figure(figsize=(9, 5))
sns.barplot(x=[CLASS_NAMES[u] for u in unique], y=counts, palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.xlabel("Class")
plt.ylabel("Number of Training Samples")
plt.title("Class Distribution (Training Set)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/02_class_distribution.png", dpi=200)
plt.close()

# =======================================================================
# TASK 2: DATA PREPROCESSING
# =======================================================================
print("\n" + "=" * 70)
print("TASK 2: Data Preprocessing")
print("=" * 70)

print(f"Before preprocessing -> x_train: {x_train_raw.shape}, dtype: {x_train_raw.dtype}")

# Normalize pixel values to [0, 1]
x_train = x_train_raw.astype("float32") / 255.0
x_test = x_test_raw.astype("float32") / 255.0

# Flatten 28x28 images into 784-length vectors
x_train = x_train.reshape(x_train.shape[0], 28 * 28)
x_test = x_test.reshape(x_test.shape[0], 28 * 28)

# One-hot encode labels
y_train = keras.utils.to_categorical(y_train_raw, num_classes=10)
y_test = keras.utils.to_categorical(y_test_raw, num_classes=10)

print(f"After preprocessing  -> x_train: {x_train.shape}, dtype: {x_train.dtype}")
print(f"One-hot y_train shape: {y_train.shape}")

# =======================================================================
# TASK 3: MODEL CONSTRUCTION
# =======================================================================
print("\n" + "=" * 70)
print("TASK 3: Model Construction (Baseline MLP)")
print("=" * 70)

def build_baseline_model():
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam",
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

baseline_model = build_baseline_model()
baseline_model.summary()

# =======================================================================
# TASK 4: MODEL TRAINING
# =======================================================================
print("\n" + "=" * 70)
print("TASK 4: Model Training (Baseline)")
print("=" * 70)

start_time = time.time()
history = baseline_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=32,
    verbose=2
)
baseline_train_time = time.time() - start_time
print(f"Baseline training time: {baseline_train_time:.2f} seconds")

# --- Plot 3 & 4: Training / Validation Accuracy vs Epoch ---
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Accuracy vs Epoch")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/03_training_accuracy.png", dpi=200)
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(history.history["val_accuracy"], label="Validation Accuracy", color="orange")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy vs Epoch")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/04_validation_accuracy.png", dpi=200)
plt.close()

# --- Plot 5 & 6: Training / Validation Loss vs Epoch ---
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss vs Epoch")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/05_training_loss.png", dpi=200)
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(history.history["val_loss"], label="Validation Loss", color="orange")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Validation Loss vs Epoch")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/06_validation_loss.png", dpi=200)
plt.close()

# =======================================================================
# TASK 5: MODEL EVALUATION (BASELINE)
# =======================================================================
print("\n" + "=" * 70)
print("TASK 5: Model Evaluation (Baseline)")
print("=" * 70)

y_pred_probs = baseline_model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

baseline_acc = accuracy_score(y_true, y_pred)
baseline_prec = precision_score(y_true, y_pred, average="macro")
baseline_rec = recall_score(y_true, y_pred, average="macro")
baseline_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Baseline Test Accuracy : {baseline_acc:.4f}")
print(f"Baseline Precision     : {baseline_prec:.4f}")
print(f"Baseline Recall        : {baseline_rec:.4f}")
print(f"Baseline F1-score      : {baseline_f1:.4f}")
print("\nClassification Report (Baseline):")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# --- Plot 7: Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (Baseline Model)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/07_confusion_matrix.png", dpi=200)
plt.close()

# =======================================================================
# SECTION 7: HYPERPARAMETER OPTIMIZATION
# (Manual Randomized Search + K-Fold CV, pure Keras -- no SciKeras.
#  SciKeras is unmaintained against newer scikit-learn releases and
#  raises "AttributeError: 'super' object has no attribute
#  '__sklearn_tags__'" with scikit-learn >= 1.6. Doing the search
#  manually avoids that dependency entirely and is more transparent.)
# =======================================================================
print("\n" + "=" * 70)
print("Hyperparameter Optimization (Manual Randomized Search, 5-Fold CV)")
print("=" * 70)

def build_search_model(hidden_layers=1, hidden_neurons=64, learning_rate=0.001,
                        activation="relu", dropout_rate=0.0, optimizer_name="adam"):
    model = keras.Sequential()
    model.add(layers.Input(shape=(784,)))
    for _ in range(hidden_layers):
        model.add(layers.Dense(hidden_neurons, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(10, activation="softmax"))

    if optimizer_name == "sgd":
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer_name == "rmsprop":
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss="categorical_crossentropy",
                   metrics=["accuracy"])
    return model

# NOTE: For classroom / laptop runtime, the search space below is a
# representative subset of the full space given in the lab manual.
# Extend these lists to the full table if you have GPU access / more time.
param_distributions = {
    "hidden_layers": [1, 2, 3],
    "hidden_neurons": [32, 64, 128, 256],
    "learning_rate": [0.1, 0.01, 0.001],
    "activation": ["relu", "tanh", "sigmoid"],
    "dropout_rate": [0.0, 0.2, 0.5],
    "batch_size": [16, 32, 64, 128],
    "epochs": [10, 20, 30],
    "optimizer_name": ["sgd", "adam", "rmsprop"],
}

N_ITER = 15          # number of random combinations to try; increase if time permits
N_FOLDS = 5
SEARCH_SAMPLE = 6000  # subset of training data to keep search time reasonable

idx = np.random.choice(len(x_train), SEARCH_SAMPLE, replace=False)
x_search, y_search = x_train[idx], y_train[idx]

# Randomly sample N_ITER hyperparameter combinations (equivalent to
# sklearn's RandomizedSearchCV sampling behaviour)
sampled_configs = list(ParameterSampler(
    param_distributions, n_iter=N_ITER, random_state=SEED
))

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

search_scores = []       # mean CV accuracy for each sampled config
search_results_log = []  # (config, mean_score) pairs

for i, config in enumerate(sampled_configs):
    fold_accuracies = []
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(x_search)):
        fold_model = build_search_model(
            hidden_layers=config["hidden_layers"],
            hidden_neurons=config["hidden_neurons"],
            learning_rate=config["learning_rate"],
            activation=config["activation"],
            dropout_rate=config["dropout_rate"],
            optimizer_name=config["optimizer_name"],
        )
        fold_model.fit(
            x_search[train_idx], y_search[train_idx],
            epochs=config["epochs"],
            batch_size=config["batch_size"],
            verbose=0,
        )
        _, fold_acc = fold_model.evaluate(
            x_search[val_idx], y_search[val_idx], verbose=0
        )
        fold_accuracies.append(fold_acc)

    mean_acc = float(np.mean(fold_accuracies))
    search_scores.append(mean_acc)
    search_results_log.append((config, mean_acc))
    print(f"[{i+1}/{N_ITER}] mean CV accuracy = {mean_acc:.4f} | config = {config}")

best_idx = int(np.argmax(search_scores))
best_params = sampled_configs[best_idx]
best_cv_accuracy = search_scores[best_idx]

print(f"\nBest CV Accuracy   : {best_cv_accuracy:.4f}")
print(f"Best Hyperparameters: {best_params}")

# --- Plot 8: Hyperparameter Search Results ---
plt.figure(figsize=(9, 5))
plt.plot(search_scores, marker="o")
plt.axvline(best_idx, color="red", linestyle="--", label="Best combination")
plt.xlabel("Hyperparameter Combination Index")
plt.ylabel("Mean CV Accuracy")
plt.title("Hyperparameter Search Results (Manual Randomized Search)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/08_hyperparameter_search.png", dpi=200)
plt.close()

# =======================================================================
# RETRAIN WITH BEST HYPERPARAMETERS ON FULL TRAINING SET
# =======================================================================
optimized_model = build_search_model(
    hidden_layers=best_params["hidden_layers"],
    hidden_neurons=best_params["hidden_neurons"],
    learning_rate=best_params["learning_rate"],
    activation=best_params["activation"],
    dropout_rate=best_params["dropout_rate"],
    optimizer_name=best_params["optimizer_name"],
)

start_time = time.time()
optimized_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=best_params["epochs"],
    batch_size=best_params["batch_size"],
    verbose=2
)
optimized_train_time = time.time() - start_time

y_pred_opt_probs = optimized_model.predict(x_test)
y_pred_opt = np.argmax(y_pred_opt_probs, axis=1)

optimized_acc = accuracy_score(y_true, y_pred_opt)
optimized_prec = precision_score(y_true, y_pred_opt, average="macro")
optimized_rec = recall_score(y_true, y_pred_opt, average="macro")
optimized_f1 = f1_score(y_true, y_pred_opt, average="macro")

print(f"\nOptimized Test Accuracy : {optimized_acc:.4f}")
print(f"Optimized Precision     : {optimized_prec:.4f}")
print(f"Optimized Recall        : {optimized_rec:.4f}")
print(f"Optimized F1-score      : {optimized_f1:.4f}")

# --- Plot 9: Best Model Accuracy Comparison ---
plt.figure(figsize=(6, 5))
bars = plt.bar(["Baseline", "Optimized"], [baseline_acc, optimized_acc],
                color=["steelblue", "seagreen"])
plt.ylabel("Test Accuracy")
plt.title("Best Model Accuracy Comparison")
plt.ylim(0, 1)
for bar, val in zip(bars, [baseline_acc, optimized_acc]):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f"{val:.3f}",
              ha="center")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/09_accuracy_comparison.png", dpi=200)
plt.close()

# =======================================================================
# FINAL SUMMARY
# =======================================================================
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"{'Metric':<15}{'Baseline':<12}{'Optimized':<12}")
print(f"{'Accuracy':<15}{baseline_acc:<12.4f}{optimized_acc:<12.4f}")
print(f"{'Precision':<15}{baseline_prec:<12.4f}{optimized_prec:<12.4f}")
print(f"{'Recall':<15}{baseline_rec:<12.4f}{optimized_rec:<12.4f}")
print(f"{'F1-score':<15}{baseline_f1:<12.4f}{optimized_f1:<12.4f}")
print(f"{'Train Time(s)':<15}{baseline_train_time:<12.2f}{optimized_train_time:<12.2f}")
print(f"\nBest Hyperparameters: {best_params}")
print(f"\nAll plots saved to the '{FIG_DIR}/' directory for inclusion in the report.")

TASK 1: Dataset Exploration
Training images shape : (60000, 28, 28)
Training labels shape : (60000,)
Testing images shape  : (10000, 28, 28)
Testing labels shape  : (10000,)
Number of classes     : 10


/tmp/ipykernel_2734/1471833464.py:78: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=[CLASS_NAMES[u] for u in unique], y=counts, palette="viridis")



TASK 2: Data Preprocessing
Before preprocessing -> x_train: (60000, 28, 28), dtype: uint8
After preprocessing  -> x_train: (60000, 784), dtype: float32
One-hot y_train shape: (60000, 10)

TASK 3: Model Construction (Baseline MLP)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)


TASK 4: Model Training (Baseline)
Epoch 1/20
1688/1688 - 26s - 15ms/step - accuracy: 0.8208 - loss: 0.5024 - val_accuracy: 0.8475 - val_loss: 0.4064
Epoch 2/20
1688/1688 - 7s - 4ms/step - accuracy: 0.8644 - loss: 0.3710 - val_accuracy: 0.8597 - val_loss: 0.3743
Epoch 3/20
1688/1688 - 8s - 5ms/step - accuracy: 0.8774 - loss: 0.3340 - val_accuracy: 0.8682 - val_loss: 0.3639
Epoch 4/20
1688/1688 - 9s - 5ms/step - accuracy: 0.8841 - loss: 0.3101 - val_accuracy: 0.8633 - val_loss: 0.3724
Epoch 5/20
1688/1688 - 8s - 5ms/step - accuracy: 0.8924 - loss: 0.2913 - val_accuracy: 0.8663 - val_loss: 0.3691
Epoch 6/20
1688/1688 - 10s - 6ms/step - accuracy: 0.8979 - loss: 0.2761 - val_accuracy: 0.8723 - val_loss: 0.3539
Epoch 7/20
1688/1688 - 9s - 5ms/step - accuracy: 0.9017 - loss: 0.2632 - val_accuracy: 0.8732 - val_loss: 0.3583
Epoch 8/20
1688/1688 - 8s - 4ms/step - accuracy: 0.9059 - loss: 0.2519 - val_accuracy: 0.8772 - val_loss: 0.3525
Epoch 9/20
1688/1688 - 6s - 4ms/step - accuracy: 0.9111 - 

In [7]:
import shutil

# Create a zip archive of the 'figures' directory
zip_filename = 'fashion_mnist_plots'
shutil.make_archive(zip_filename, 'zip', FIG_DIR)

print(f"Plots saved to: {zip_filename}.zip")

Plots saved to: fashion_mnist_plots.zip


In [ ]:
from google.colab import files

# Download the zip file
files.download(zip_filename + '.zip')